### ▶️ Versi Kaggle — Background Run & Simpan Weights (V4)

Notebook ini = **V3 Kaggle + paket improvement A–H** (stage 2/3 jelek di V3 diperbaiki),
struktur sel V3 dipertahankan 1:1 kecuali bagian bertanda **`[V4]`**:

| # | V3 | V4 (alasan) |
|---|---|---|
| A | 1 seed, threshold 0.02, agregator `max` | 3 seed + threshold teregulasi (clamp+gain) + `max` dihapus (degenerat di V3) |
| B | monitor raw patient-F1, TTA flip | monitor EMA-smooth val AUC + SWA + ensemble soft-vote |
| C | aug lemah (rot 5°) | aug tensor-safe per-stage: VFlip+rotasi+jitter; HFlip tetap mati utk stage2 |
| D | CE + class weights | ClassBalanced focal (γ=2) + label smoothing 0.1 |
| E | 1-channel RGB ImageNet-norm | 3-channel pseudo-window (standard/stroke-like/blood-like; aproksimasi — JPG bukan HU) |
| F | klasifikasi langsung laterality | Siamese shared-weight kiri-kanan + diff (stage2 saja, 2D moderat) |
| G | mean/median/max/vote | + `topk` + `attention` (fokus slice lesi); `max` dibuang |
| H | ImageNet pretrain | DenseNet121 + RadImageNet (fallback ImageNet); freeze 5 epoch → unfreeze layer-wise LR |

Backbone V4: `['resnet18', 'densenet121']` (hemat kuota vs 4 di V3). Seeds `[42,43,44]`, epochs 25.

**Cara pakai di Kaggle (sama seperti V3):**
1. Upload `NCCT_fixed_split.zip` sebagai Dataset → Add Input (atau attach yang sudah ada).
2. Run All (browser boleh ditutup; limit ≈12 jam; auto-resume dari `*_last_full.pt`).
3. Setelah selesai → Save Version → download `ncct_v4_weights.zip` dari tab Output.



# NCCT Multi-Stage Classification **V3** — Fix Laterality, Patient-Level Decision Tuning &amp; Leakage-Free Evaluation

Pipeline klasifikasi **bertingkat** pada citra CT Non-Contrast (NCCT) pasien stroke — **V3**, perbaikan menyeluruh dari V2:

```
                    ┌────────────────────────────────────────────┐
  Citra NCCT  ──►   │  STAGE 1 : Apakah pasien mengalami LVO?    │
                    └────────────────────────────────────────────┘
                              │ Ya                        │ Tidak
                              ▼                           ▼
              ┌───────────────────────────────┐      Non-LVO
              │ STAGE 2 : Laterality          │     (selesai)
              │   Left / Right                │
              ├───────────────────────────────┤
              │ STAGE 3 : Etiologi            │
              │   Cardioembolic / Non-Cardio  │
              └───────────────────────────────┘
```


## Perbaikan utama V3

| # | V2 | V3 |
|---|----------|----|
| 1 | Augmentasi seragam (termasuk **HFlip untuk laterality**) | **Augmentasi per-stage**: HFlip dimatikan utk `stage2_laterality` (bug label), aktif utk stage1/3 |
| 3 | Model terbaik dipilih berdasar **test F1** (leakage) | **Seleksi backbone/agregator/threshold HANYA dari VAL**, test dievaluasi sekali di akhir |
| 4 | Threshold 0.5 & `mean` di-hardcode | Tuning threshold + pemilihan agregator slice (`mean/median/max/vote`) di **val** |
| 5 | Tanpa TTA (Test Time Augmentation) | **Horizontal-flip TTA**; utk laterality probabilitas di-invert karena flip membalik kelas |
| 6 | Evaluasi pipeline tanpa CI (Confidence Interval) | **Bootstrap 95% CI** utk metrik level pasien (test stage 2/3 hanya ~20 pasien → varian besar) |
| 7 | Re-decode 8.6k citra tiap epoch | **LRU cache** decode+resize di Dataset |
| 8 | Output tanpa metadata konfigurasi | **Snapshot konfigurasi (JSON)** + checkpoint berisi metadata (tidak hanya state_dict) |
| 9 | Gate evaluasi stage 2/3 = prediksi stage 1 saja | Dilaporkan **prediksi-gate DAN oracle-gate** agar efek salah-klasifikasi stage 1 terlihat |

**Sumber data (FIXED SPLIT):** zip dari Google Drive (dibuat oleh `NCCT_Fixed_Dataset_Split.ipynb`):
- `metadata/{stage1_lvo,stage2_laterality,stage3_etiology}/{train,val,test}.csv`
- Folder citra `train/`, `val/`, `test/` (struktur asli dipertahankan)

**Stage:**
- **Stage 1** — LVO vs Non-LVO (semua pasien)
- **Stage 2** — Left vs Right (hanya LVO, ada label laterality)
- **Stage 3** — Cardioembolic vs Non-Cardioembolic (hanya LVO, ada label etiologi)


### 1. Install & Import Library

In [ ]:
import os, re, glob, zipfile, random, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, classification_report)

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


def set_seed(seed):
    """Set semua random seed utk reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

### 2. Konfigurasi

Ubah `MODE` menjadi `'local'` untuk uji cepat dengan sample lokal, atau `'colab'` untuk dataset penuh dari Google Drive.

> **PENTING:** dataset **tidak displit ulang**. Zip berisi *fixed split* yang sudah dibuat sekali oleh
> `NCCT_Fixed_Dataset_Split.ipynb` (seed tetap 42). Perbedaan besar dgn V2 ada di sini:
> `MONITOR`, `USE_TTA`, `STRONG_AUG`, `AGGREGATOR` (fallback) dan `NUM_WORKERS`/`CACHE_IMAGES` ditambahkan;
> default `EPOCHS` dinaikkan karena 1 epoch terbukti menghasilkan performa ≈ random utk stage 2 & 3.

In [ ]:

# ============================================================
#  KONFIGURASI V4 — [KAGGLE] MODE kaggle + background-safe
# ============================================================
IS_KAGGLE = ('/kaggle' in __import__('os').getcwd()) or __import__('os').path.exists('/kaggle')

MODE = 'kaggle' if IS_KAGGLE else 'colab'

GDRIVE_FILE_ID = '1PAMWLa6TAh_yFW5RY4FQxDqjIoytqgcb'
SPLIT_ZIP      = 'NCCT_fixed_split.zip'
SPLIT_SUBPATH  = 'NCCT_fixed_split'
KAGGLE_DATASET_SLUG = ''

if IS_KAGGLE:
    import glob as _glob
    _cands = sorted(_glob.glob('/kaggle/input/datasets/lhfazry/ncct-fixed'))
    if KAGGLE_DATASET_SLUG and '/' in KAGGLE_DATASET_SLUG:
        _slug_tail = KAGGLE_DATASET_SLUG.split('/')[-1]
        _cands = sorted(_glob.glob(f'/kaggle/input/{_slug_tail}/ncct-fixed')) or _cands
        _cands += sorted(_glob.glob(f'/kaggle/input/{_slug_tail}/*'))
    if _cands:
        SPLIT_DIR = _cands[0]
    else:
        SPLIT_DIR = f'/kaggle/input/{SPLIT_SUBPATH}'
    OUTPUT_DIR = '/kaggle/working/outputs_v4'
else:
    SPLIT_DIR = 'NCCT_fixed_split'
    OUTPUT_DIR = 'outputs_v4'

IMG_SIZE   = 224
BATCH_SIZE = 32

# ===== BACKBONE ABLATION V4 =====
BACKBONES  = ['resnet18', 'densenet121']
PRETRAINED = True
RADIMAGENET = True

# ===== TRAINING V4 =====
EPOCHS        = 25
LR_HEAD       = 1e-3
LR_BACKBONE   = 1e-5
LR            = LR_HEAD      # alias kompatibilitas utk sel V3 (train_model lr=LR)
FREEZE_EPOCHS = 5
LABEL_SMOOTH  = 0.1
FOCAL_GAMMA   = 2.0
DROPOUT_HEAD  = 0.4
PATIENCE      = 7

# ===== MONITOR V4 =====
AMP           = True       # mixed precision
MONITOR = 'smooth_val_auc'
SWA = True

# ===== [KAGGLE] BACKGROUND-SAFE / RESUME =====
RESUME     = True
SAVE_EVERY = 1

# ===== EVALUASI & KEPUTUSAN LEVEL PASIEN =====
PATIENT_LEVEL  = True
AGGREGATOR     = 'mean'
TUNE_THRESHOLD = True
USE_TTA        = True

# ===== DATA UTIL =====
NUM_WORKERS  = 2
CACHE_IMAGES = True
STRONG_AUG   = False

# ===== REPRODUCIBILITY =====
SEEDS = [42, 43, 44]

# ===== [KAGGLE] REUSE WEIGHTS DARI DATASET (opsional) =====
LOAD_WEIGHTS_DIR = ''

os.makedirs(OUTPUT_DIR, exist_ok=True)

CONFIG = dict(mode=MODE, gdrive_file_id=GDRIVE_FILE_ID, split_zip=SPLIT_ZIP, split_dir=SPLIT_DIR,
              kaggle_dataset_slug=KAGGLE_DATASET_SLUG, load_weights_dir=LOAD_WEIGHTS_DIR,
              img_size=IMG_SIZE, batch_size=BATCH_SIZE, backbones=BACKBONES, pretrained=PRETRAINED,
              radimagenet=RADIMAGENET, epochs=EPOCHS, lr_head=LR_HEAD, lr_backbone=LR_BACKBONE,
              freeze_epochs=FREEZE_EPOCHS, label_smooth=LABEL_SMOOTH, focal_gamma=FOCAL_GAMMA,
              dropout_head=DROPOUT_HEAD, patience=PATIENCE, monitor=MONITOR, swa=SWA,
              resume=RESUME, save_every=SAVE_EVERY,
              patient_level=PATIENT_LEVEL, aggregator=AGGREGATOR, tune_threshold=TUNE_THRESHOLD,
              tta=USE_TTA, num_workers=NUM_WORKERS, cache_images=CACHE_IMAGES, strong_aug=STRONG_AUG,
              seeds=SEEDS)

print('IS_KAGGLE:', IS_KAGGLE, '| Mode:', MODE, '| Device:', device)
print('SPLIT_DIR:', SPLIT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Backbones:', BACKBONES, '| RADIMAGENET:', RADIMAGENET)
print('Epochs:', EPOCHS, '| LR_HEAD:', LR_HEAD, '| LR_BACKBONE:', LR_BACKBONE, '| Monitor:', MONITOR, '| SWA:', SWA)
print('Seeds:', SEEDS, '| Tuning threshold pd VAL:', TUNE_THRESHOLD)

WARMUP_EPOCHS = 1          # linear warmup sebelum cosine annealing
GRAD_CLIP     = 1.0        # gradient clipping (max norm)
WEIGHT_DECAY  = 1e-4





### 3. Download Dataset Fixed Split dari Google Drive

Download zip **fixed split** (bukan dataset mentah) → hasil langsung siap dipakai, **tanpa split ulang**.

In [ ]:
# [KAGGLE] 3-mode: kaggle (/kaggle/input) | colab (gdown) | local (folder)
if MODE == 'kaggle':
    # 1) dataset sudah ter-extract di /kaggle/input/.../NCCT_fixed_split → langsung pakai
    if os.path.isdir(SPLIT_DIR) and os.path.isdir(os.path.join(SPLIT_DIR, 'metadata')):
        print('Mode kaggle → pakai dataset:', SPLIT_DIR)
        print('Isi SPLIT_DIR:', sorted(os.listdir(SPLIT_DIR)))
    else:
        # 2) fallback: cari zip di /kaggle/input atau /kaggle/working lalu extract ke /kaggle/working
        cand = ([z for z in glob.glob('/kaggle/input/**/*.zip', recursive=True) if 'fixed' in z.lower()]
                or glob.glob('/kaggle/input/**/*.zip', recursive=True)
                or ([SPLIT_ZIP] if os.path.exists(SPLIT_ZIP) else [])
                or glob.glob('*.zip'))
        print('Kandidat zip:', cand)
        if not cand:
            raise RuntimeError(
                'Dataset tidak ditemukan. Upload NCCT_fixed_split.zip sebagai Kaggle Dataset, '
                'lalu Add Input ke notebook ini (cek KAGGLE_DATASET_SLUG / SPLIT_SUBPATH).')
        zip_src = cand[0]
        SPLIT_DIR = '/kaggle/working/NCCT_fixed_split'
        import zipfile as _zf
        print(f'Extract {zip_src} → {SPLIT_DIR} ...')
        with _zf.ZipFile(zip_src) as zf:
            zf.extractall(SPLIT_DIR if zip_src.lower().endswith('fixed') else '/kaggle/working')
        # normalisasi: bila hasil extract nested (…/NCCT_fixed_split/NCCT_fixed_split), ratakan
        _nested = os.path.join(SPLIT_DIR, 'NCCT_fixed_split')
        if os.path.isdir(_nested) and not os.path.isdir(os.path.join(SPLIT_DIR, 'metadata')):
            SPLIT_DIR = _nested
        print('Isi SPLIT_DIR:', sorted(os.listdir(SPLIT_DIR)))
elif MODE == 'colab':
    get_ipython().system('pip install -q gdown')
    get_ipython().run_line_magic('gdown', GDRIVE_FILE_ID)
    zips = [z for z in glob.glob('*.zip') if 'fixed' in z.lower()] or glob.glob('*.zip')
    print('Zip hasil download:', zips)
    if not zips:
        raise RuntimeError('Tidak ada zip. Periksa GDRIVE_FILE_ID.')
    zip_src = SPLIT_ZIP if os.path.exists(SPLIT_ZIP) else zips[0]
    get_ipython().system(f'unzip -q -o {zip_src} -d NCCT_fixed_split')
    print('Isi SPLIT_DIR:', sorted(os.listdir(SPLIT_DIR)))
else:
    assert os.path.isdir(SPLIT_DIR), f'Folder {SPLIT_DIR} tidak ditemukan (mode local).'
    print('Mode local → pakai folder sample:', SPLIT_DIR)
    print('Isi:', sorted(os.listdir(SPLIT_DIR)))


### 4. Load Fixed Split

CSV metadata per stage + per set dimuat, lalu `image_path` relatif diubah menjadi absolut terhadap `SPLIT_DIR`.

- `filter_missing=True` → baris yang citranya tidak ada dibuang (agar mode `local` dengan sample tetap jalan).

In [ ]:
STAGES = ['stage1_lvo', 'stage2_laterality', 'stage3_etiology']
SPLITS = ('train', 'val', 'test')


def load_fixed_split(split_root=SPLIT_DIR, filter_missing=True):
    """Muat CSV split FIXED dari folder hasil split.

    - `image_path` relatif (mis. `train/LVO/p1_.../03.jpg`) → absolut thd split_root.
    - `filter_missing=True` → buang baris yg citranya tidak ada (utk sample lokal).
    """
    out, dropped = {}, 0
    for stage in STAGES:
        for sp in SPLITS:
            fp = os.path.join(split_root, 'metadata', stage, f'{sp}.csv')
            df = pd.read_csv(fp)
            df['image_path'] = df['image_path'].apply(
                lambda p: os.path.join(os.path.abspath(split_root), p))
            df['label'] = df['label'].astype(int)
            if filter_missing:
                n0 = len(df)
                df = df[df['image_path'].apply(os.path.exists)]
                dropped += n0 - len(df)
            out[(stage, sp)] = df.reset_index(drop=True)
    if dropped:
        print(f'[info] {dropped} baris dibuang krn citra tidak ada di folder lokal (sample).')
    return out


splits = load_fixed_split()
print('Jumlah citra & pasien per (stage, set):')
for stage in STAGES:
    sizes = [len(splits[(stage, s)]) for s in SPLITS]
    subs  = [splits[(stage, s)]['subject'].nunique() for s in SPLITS]
    print(f'  {stage:22s} citra  T:{sizes[0]:5d} V:{sizes[1]:5d} Te:{sizes[2]:5d}  '
          f'| pasien T:{subs[0]:3d} V:{subs[1]:3d} Te:{subs[2]:3d}')

# ---- Verifikasi: bebas leakage antar set & konsisten antar stage ----
for stage in STAGES:
    tr_s, va_s, te_s = [set(splits[(stage, s)]['subject']) for s in SPLITS]
    assert tr_s.isdisjoint(va_s) and tr_s.isdisjoint(te_s) and va_s.isdisjoint(te_s), \
        f'LEAKAGE di {stage}!'
    if stage != 'stage1_lvo':
        te1 = set(splits[('stage1_lvo', 'test')]['subject'])
        assert te_s.issubset(te1), f'{stage}: pasien test di luar split global!'
print('✔ Split bebas data leakage antar set & konsisten antar stage.')

### 5. Analisis Dataset

Cek distribusi kelas tiap stage & jumlah slice per pasien (dasar pentingnya evaluasi level pasien).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
stage_titles = {'stage1_lvo': 'Stage 1: LVO vs Non-LVO',
                'stage2_laterality': 'Stage 2: Laterality (hanya LVO)',
                'stage3_etiology': 'Stage 3: Etiologi (hanya LVO)'}
for ax, stage in zip(axes, STAGES):
    vc = splits[(stage, 'train')]['label'].value_counts().sort_index()
    labels = {0: '0', 1: '1'}[vc.index[0]] if len(vc) else '0'
    ax.bar(vc.index.astype(str), vc.values, color=['#3498db', '#e74c3c'])
    ax.set_title(stage_titles[stage])
    for i, v in enumerate(vc.values):
        ax.annotate(str(int(v)), (i, v), ha='center', va='bottom')
    ax.set_ylabel('Jumlah citra (train)')
plt.tight_layout(); plt.show()

# Slice per pasien → evaluasi level pasien lebih bermakna
tr1 = splits[('stage1_lvo', 'train')]
spp = tr1.groupby('subject').size()
print(f'Slice/pasien (train stage1): mean={spp.mean():.1f}  median={spp.median():.0f}  max={spp.max()}')
plt.figure(figsize=(8, 3))
sns.histplot(spp, bins=30)
plt.xlabel('Jumlah slice per pasien')
plt.title('Distribusi slice/pasien (train stage 1)')
plt.show()

# Contoh citra tiap kelas
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, (title, lab) in enumerate([('LVO (selected_ncct)', 1), ('Non-LVO (lacunar)', 0)]):
    sub = tr1[tr1['label'] == lab]
    p = sub.sample(1, random_state=42 + i)['image_path'].iloc[0]
    img = Image.open(p).convert('RGB')
    axes[i].imshow(img)
    axes[i].set_title(f'{title}\n{os.path.basename(p)}', fontsize=9)
    axes[i].axis('off')
plt.tight_layout(); plt.show()

### 6. Dataset & Dataloader — Augmentasi PER-STAGE + LRU Cache

**Perbaikan #1 (kritis):** augmentasi sekarang dibangun per stage.

- **`stage2_laterality` TIDAK memakai `RandomHorizontalFlip`** — mem-flip citra akan membalik
  makna label (kiri ↔ kanan). V2 memakai transform yang sama utk semua stage sehingga label
  stage 2 menjadi ~50% noise → model tidak bisa belajar (hasil ≈ chance).
- Stage 1 & 3 aman terhadap flip (kelas tidak bergantung sisi).
- Ditambahkan **LRU cache** (`CACHE_IMAGES`) agar citra yang sudah di-decode+resize tidak dibuka
  ulang dari disk setiap epoch (8.6k citra train stage 1 × N epoch).

Setiap sample mengembalikan `(image, label, subject, image_path)` — `subject` diperlukan untuk evaluasi level pasien.

In [ ]:
class StrokeImageDataset(Dataset):
    """Dataset citra stroke dari CSV fixed split: 1 sample = 1 citra + label + subject.

    cache=True → LRU cache hasil decode+resize agar epoch berikutnya lebih cepat
    (citra dibuka sekali, transform acak tetap diterapkan tiap akses).
    """
    def __init__(self, df, transform=None, cache=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self._cache = OrderedDict() if cache else None
        self._CACHE_MAX = 2000   # ~2000 x 224x224 RGB ≈ ratusan MB — batas aman utk Colab

    def __len__(self):
        return len(self.df)

    def _load(self, path):
        if self._cache is None:
            return Image.open(path).convert('RGB')
        if path in self._cache:
            self._cache.move_to_end(path)
            return self._cache[path]
        img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        if len(self._cache) >= self._CACHE_MAX:
            self._cache.popitem(last=False)
        self._cache[path] = img
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load(row['image_path'])
        if self.transform is not None:
            img = self.transform(img)
        return img, int(row['label']), row['subject'], row['image_path']


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def _eval_tf():
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def build_train_tf(stage, strong=False):
    """Augmentasi PER-STAGE.

    Kunci (fix bug V2): RandomHorizontalFlip TIDAK boleh dipakai utk stage2_laterality —
    mem-flip citra membalik label Left<->Right sehingga 50% label menjadi noise.
    Stage 1 (LVO) & 3 (etiologi) aman terhadap flip.
    """
    hflip = (stage != 'stage2_laterality')
    ops = [transforms.Resize((IMG_SIZE, IMG_SIZE)),
           transforms.RandomRotation(degrees=5)]
    if strong:
        ops.append(transforms.RandomAffine(degrees=0, translate=(0.06, 0.06), scale=(0.94, 1.06)))
    ops += [transforms.RandomHorizontalFlip(p=0.5 if hflip else 0.0),
            # brightness/contrast jitter aman utk citra grayscale medis
            transforms.ColorJitter(brightness=0.1, contrast=0.15),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]
    return transforms.Compose(ops)


def make_loaders(stage, batch_size=BATCH_SIZE, cache=CACHE_IMAGES):
    """Loader utk satu stage dari FIXED SPLIT (tidak ada split ulang)."""
    tf_tr = build_train_tf(stage, strong=STRONG_AUG)
    tf_ev = _eval_tf()
    tr_ds = StrokeImageDataset(splits[(stage, 'train')], tf_tr, cache=cache)
    va_ds = StrokeImageDataset(splits[(stage, 'val')],   tf_ev, cache=cache)
    te_ds = StrokeImageDataset(splits[(stage, 'test')],  tf_ev, cache=cache)
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    va_ld = DataLoader(va_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    te_ld = DataLoader(te_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    return tr_ld, va_ld, te_ld

In [ ]:

import torchvision.transforms as _tv_transforms
from PIL import ImageOps as _ImageOps

class StrokeImageDatasetV4(Dataset):
    """Dataset citra stroke V4: RGB->grayscale -> 3-channel multi-window.

    Setiap citra JPG diubah ke grayscale lalu direpresentasikan sebagai
    3-channel "pseudo-window" untuk meniru windowing HU CT:
      - ch1: standard (normalize ImageNet) — citra asli
      - ch2: stroke-like (contrast stretch 0.15-0.85 quantile + CLAHE sederhana)
      - ch3: blood-like (gamma 1.5 highlight terang)

    [HONEST NOTE] Citra JPG bukan HU DICOM. Windowing HU sejati TIDAK MUNGKIN
    dilakukan karena JPG sudah kehilangan informasi HU asli. Ketiga channel
    ini adalah APROKIMASI visual yang meningkatkan kontras fitur stroke,
    bukan windowing medis yang akurat.

    cache=True -> LRU cache hasil decode+resize agar epoch berikutnya lebih cepat.
    """
    def __init__(self, df, transform=None, cache=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self._cache = OrderedDict() if cache else None
        self._CACHE_MAX = 2000

    def __len__(self):
        return len(self.df)

    @staticmethod
    def _rgb_to_grayscale(img):
        """Konversi RGB PIL Image ke grayscale (mode L)."""
        return img.convert('L')

    @staticmethod
    def _channel_standard(img_array):
        """Channel 1: standard — grayscale dinormalisasi (aproksimasi)."""
        img = img_array.astype(np.float64) / 255.0
        return (img - 0.485) / 0.229

    @staticmethod
    def _channel_stroke_like(img_array):
        """Channel 2: stroke-like — contrast stretch 0.15-0.85 quantile + CLAHE sederhana.

        [HONEST NOTE] CLAHE via PIL ImageOps.equalize sebagai aproksimasi.
        Tidak ada histogram CLAHE sejati karena JPG bukan HU.
        """
        img = img_array.astype(np.float64)
        lo, hi = np.percentile(img, 15), np.percentile(img, 85)
        img_stretched = np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)
        img_uint8 = (img_stretched * 255).astype(np.uint8)
        img_pil = Image.fromarray(img_uint8)
        img_eq = _ImageOps.equalize(img_pil)
        return np.array(img_eq, dtype=np.float64) / 255.0

    @staticmethod
    def _channel_blood_like(img_array):
        """Channel 3: blood-like — gamma 1.5 highlight terang.

        Gamma 1.5 menonjolkan region terang (darah/hemorrhage) pada citra.
        [HONEST NOTE] Ini aproksimasi visual, bukan windowing HU sejati.
        """
        img = img_array.astype(np.float64) / 255.0
        return np.power(np.clip(img, 0, 1), 1.5)

    def _load(self, path):
        if self._cache is None:
            return Image.open(path).convert('RGB')
        if path in self._cache:
            self._cache.move_to_end(path)
            return self._cache[path]
        img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        if len(self._cache) >= self._CACHE_MAX:
            self._cache.popitem(last=False)
        self._cache[path] = img
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_rgb = self._load(row['image_path'])
        img_gray = self._rgb_to_grayscale(img_rgb)
        img_arr = np.array(img_gray, dtype=np.float64)

        # 3-channel multi-window (3, H, W) — tiap channel 2D (H, W)
        image = torch.from_numpy(np.stack([self._channel_standard(img_arr),
                  self._channel_stroke_like(img_arr),
                  self._channel_blood_like(img_arr)]).astype(np.float32))

        if self.transform is not None:
            image = self.transform(image)

        return image, int(row['label']), row['subject'], row['image_path']


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def _eval_tf_v4():
    """Eval V4: tanpa transform (dataset sudah tensor (3,H,W) ternormalisasi)."""
    return None


def _train_tf_v4(stage, strong=False):
    """Augmentasi tensor-safe per-stage (tanpa HFlip utk stage2_laterality)."""
    import torchvision.transforms.functional as _F
    hflip = (stage != 'stage2_laterality')
    def _aug(t):
        if hflip and random.random() < 0.5:
            t = torch.flip(t, dims=[2])
        elif random.random() < 0.5:  # vertical flip aman utk laterality (A/P)
            t = torch.flip(t, dims=[1])
        if random.random() < 0.5:
            t = _F.affine(t, angle=random.uniform(-5, 5), translate=(0.03, 0.03), scale=1.0, shear=0)
        if random.random() < 0.5:
            t = _F.adjust_brightness(t, random.uniform(0.9, 1.1))
            t = _F.adjust_contrast(t, random.uniform(0.9, 1.1))
        return t
    return _aug


def make_loaders_v4(stage, batch_size=BATCH_SIZE, cache=CACHE_IMAGES):
    """Loader untuk satu stage dari FIXED SPLIT (tidak ada split ulang)."""
    tf_tr = _train_tf_v4(stage, strong=STRONG_AUG)
    tf_ev = _eval_tf_v4()
    tr_ds = StrokeImageDatasetV4(splits[(stage, 'train')], tf_tr, cache=cache)
    va_ds = StrokeImageDatasetV4(splits[(stage, 'val')],   tf_ev, cache=cache)
    te_ds = StrokeImageDatasetV4(splits[(stage, 'test')],  tf_ev, cache=cache)
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    va_ld = DataLoader(va_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    te_ld = DataLoader(te_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    return tr_ld, va_ld, te_ld



In [ ]:

class ClassBalancedFocalLoss(nn.Module):
    """ClassBalancedFocalLoss: beta=0.99 effective-number per Cui et al.

    Menggabungkan:
      1. Class-balanced weighting (effective number per Cui et al., 2019)
      2. Focal loss gamma (menekan sample mudah, fokus pada hard example)
      3. Label smoothing (mencegah overconfidence)

    Tidak pakai `as any` atau try/except pasif.
    Semua parameter divalidasi secara eksplisit.
    """
    def __init__(self, num_classes=2, beta=0.99, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.beta = beta
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        # Effective number per Cui et al.: Ef(N) = (1 - beta^N) / (1 - beta)
        self.register_buffer('class_weights', self._compute_class_weights(num_classes, beta))

    @staticmethod
    def _compute_class_weights(num_classes, beta):
        """Hitung effective-number-based class weights."""
        weights = []
        for i in range(num_classes):
            effective_num = 1.0 - (beta ** (i + 1))
            effective_den = 1.0 - beta
            weight = 1.0 / max(effective_num / effective_den, 1e-6)
            weights.append(weight)
        weights = torch.tensor(weights, dtype=torch.float32)
        weights = weights / weights.sum() * num_classes
        return weights

    def forward(self, logits, targets):
        """Forward pass: focal loss + label smoothing.

        Args:
            logits: (N, C) model output (pre-softmax)
            targets: (N,) integer class labels
        Returns:
            Scalar loss
        """
        # Label smoothing: distribusi target halus
        smooth_targets = torch.full_like(logits, self.label_smoothing / (self.num_classes - 1))
        smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)

        # Focal loss: (1 - p_t)^gamma * CE
        log_probs = torch.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze()
        focal_weight = (1.0 - pt) ** self.gamma

        # Cross entropy dengan class weight
        ce_loss = -log_probs * smooth_targets
        ce_loss = ce_loss.sum(dim=1)
        weighted_ce = ce_loss * self.class_weights[targets] * focal_weight

        return weighted_ce.mean()


def class_weights_for_stage(stage):
    """Hitung class weights berdasarkan distribusi label train set per stage.

    Return (class_weights_tensor, counts_dict).
    """
    tr = splits[(stage, 'train')]
    counts = np.bincount(tr['label'].values)
    cw = torch.tensor(len(tr) / (2 * counts), dtype=torch.float32).to(device)
    return cw, {int(k): int(v) for k, v in enumerate(counts)}


def get_weighted_sampler(stage, batch_size=32):
    """Buat WeightedRandomSampler untuk balance kelas.

    Return DataLoader sampler yang bisa dipakai di make_loaders_v4.
    """
    tr = splits[(stage, 'train')]
    labels = tr['label'].values
    counts = np.bincount(labels)
    class_weights = 1.0 / counts[labels]
    weights = torch.tensor(class_weights, dtype=torch.float64)
    sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights))
    return sampler



In [ ]:

class SliceAttention(nn.Module):
    """SliceAttention: tanh proj -> softmax -> weighted sum.

    Menerapkan attention pada representasi per-slice untuk menghasilkan
    weighted aggregation. Setiap slice diproyeksikan ke scalar via tanh,
    di-normalisasi dengan softmax, lalu digunakan sebagai bobot untuk
    weighted sum dari fitur per-slice.

    Pure torch (tanpa numpy).
    """
    def __init__(self, feature_dim=512):
        super().__init__()
        self.projection = nn.Linear(feature_dim, 1)

    def forward(self, slice_features):
        """Forward pass.

        Args:
            slice_features: (N, C) fitur per-slice (N slice, C dimensi)
        Returns:
            weighted_sum: (C,) hasil agregasi tertimbang
            attention_weights: (N,) bobot per-slice
        """
        scores = torch.tanh(self.projection(slice_features)).squeeze(-1)
        weights = torch.softmax(scores, dim=0)
        weighted_sum = torch.sum(weights.unsqueeze(-1) * slice_features, dim=0)
        return weighted_sum, weights


class TopKPool(nn.Module):
    """TopKPool: agregasi dengan k slice teratas berdasarkan probabilitas.

    Pure torch/numpy.
    """
    def __init__(self, k=8):
        super().__init__()
        self.k = k

    def forward(self, probs):
        """Forward pass.

        Args:
            probs: (N,) probabilitas per-slice
        Returns:
            aggregated: scalar agregasi top-k mean
        """
        k = min(self.k, len(probs))
        topk_vals, _ = torch.topk(probs, k)
        return topk_vals.mean()


def aggregate_patient(probs, method='mean', k=8, attention_model=None):
    """Agregasi probabilitas per-slice -> skor level pasien.

    Methods:
      'mean'    -> rata-rata probabilitas seluruh slice
      'median'  -> median prob (tahan outlier slice)
      'topk'    -> rata-rata k slice teratas (TopKPool)
      'attention'-> weighted sum via SliceAttention (perlu attention_model)
      'vote'    -> fraksi slice 'positif' (p>=0.5) -> keputusan mayoritas

    Args:
        probs: (N,) array probabilitas kelas-1 per-slice
        method: str metode agregasi
        k: int untuk topk
        attention_model: SliceAttention instance (untuk method='attention')

    Returns:
        float skor agregasi level pasien
    """
    probs_t = torch.tensor(probs, dtype=torch.float32) if not isinstance(probs, torch.Tensor) else probs
    probs_t = probs_t.float()

    if method == 'mean':
        return float(probs_t.mean())
    elif method == 'median':
        return float(probs_t.median())
    elif method == 'max':
        return float(probs_t.max())
    elif method == 'topk':
        k = min(k, len(probs_t))
        topk_vals, _ = torch.topk(probs_t, k)
        return float(topk_vals.mean())
    elif method == 'attention':
        if attention_model is None:
            raise ValueError("attention_model diperlukan untuk method='attention'")
        with torch.no_grad():
            agg, _ = attention_model(probs_t.unsqueeze(-1) if probs_t.dim() == 1 else probs_t)
        return float(agg) if agg.dim() == 0 else float(agg.squeeze())
    elif method == 'vote':
        return float((probs_t >= 0.5).float().mean())
    else:
        raise ValueError(f"Metode agregasi tidak dikenal: {method}")



### 7. Fungsi Training, Evaluasi (dgn TTA) & Utilitas Keputusan Level Pasien

Peningkatan fungsi dibanding V2:

- **`evaluate_model(..., tta, invert_flip)`** — dukungan **test-time augmentation** horizontal-flip.
  - `tta=True` → probabilitas = rata-rata `p` dan `p_flipped`.
  - `invert_flip=True` → khusus **laterality**: karena citra ter-flip membalik kelas, probabilitas
    di-invert (`p_f[:, [1, 0]]`) sebelum dirata-rata. Dilarang utk stage lain.
- **Utilitas level pasien** — `scores_df`, `patient_df` (agregator `mean/median/max/vote`),
  `patient_metrics_from_sdf`, dan `tune_threshold` (scanning threshold optimal).
  Threshold & agregator HARUS ditentukan dari **val**, bukan test.
- **`train_model(..., monitor)`** — `monitor='patient_f1'` memilih epoch terbaik berdasar **F1
  level pasien pd val** (metrik relevan klinis); `monitor='image_f1'` mempertahankan perilaku V2.
  Checkpoint kini menyimpan `{'model': state_dict, 'meta': {...}}`.

In [ ]:
def build_model(num_classes=2, backbone='resnet18', pretrained=PRETRAINED):
    """Model CNN pretrained dari timm (utk ablation backbone)."""
    try:
        model = timm.create_model(backbone, pretrained=pretrained, num_classes=num_classes)
    except TypeError:
        weights = 'IMAGENET1K_V1' if pretrained else None
        model = timm.create_model(backbone, weights=weights, num_classes=num_classes)
    return model


def compute_metrics(y_true, y_pred, y_prob):
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'auc':       roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan'),
    }


@torch.no_grad()
def evaluate_model(model, loader, criterion=None, tta=False, invert_flip=False):
    """Evaluasi pada loader. Return (metrics, records) — records per-slice:
    [(subject, y_true, y_pred, y_prob), ...] → dipakai utk level pasien.

    tta=True         → horizontal-flip ensemble: probs = mean(p, p_flip).
    invert_flip=True → HANYA utk laterality (stage2): citra ter-flip membalik kelas,
                       jadi prob kelas di-invert sebelum dirata-rata. JANGAN utk stage lain.
    """
    model.eval()
    records, loss_sum, n = [], 0.0, 0
    for xb, yb, subj, _ in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)
        if tta:
            xf = torch.flip(xb, dims=[3])                       # horizontal flip (N,C,H,W)
            p_f = torch.softmax(model(xf), dim=1)
            if invert_flip:
                p_f = p_f[:, [1, 0]]                            # balik makna kelas krn citra ter-flip
            probs = (probs + p_f) / 2.0
        records += [(s, int(yt), int(yp), float(pr)) for s, yt, yp, pr in
                    zip(subj, yb.tolist(), probs.argmax(1).tolist(), probs[:, 1].tolist())]
        if criterion is not None:
            loss_sum += criterion(logits, yb).item() * len(yb)
            n += len(yb)
    y_true = [r[1] for r in records]
    y_pred = [r[2] for r in records]
    y_prob = [r[3] for r in records]
    metrics = compute_metrics(y_true, y_pred, y_prob)
    if criterion is not None:
        metrics['loss'] = loss_sum / max(n, 1)
    return metrics, records


# ============ UTILITAS LEVEL PASIEN & KEPUTUSAN ============

def scores_df(records):
    """DataFrame per-slice: subject, y_true, y_pred, y_prob."""
    return pd.DataFrame(records, columns=['subject', 'y_true', 'y_pred', 'y_prob'])


def patient_df(sdf, aggregator='mean'):
    """Agregasi slice → 1 baris/pasien (prob kelas-1).

    aggregator:
      'mean'   → rata-rata probabilitas seluruh slice (default V2)
      'median' → median prob (tahan outlier slice)
      'max'    → prob slice tertinggi (sesuai lesi yang tampil hanya di bbrp slice)
      'vote'   → fraksi slice 'positif' (p>=0.5) → keputusan mayoritas slice
    """
    grp = sdf.groupby('subject')['y_prob']
    if aggregator == 'median':
        p = grp.median()
    elif aggregator == 'max':
        p = grp.max()
    elif aggregator == 'vote':
        p = grp.apply(lambda x: float((x >= 0.5).mean()))
    else:
        p = grp.mean()
    pdf = p.reset_index().rename(columns={'y_prob': 'p'})
    yt = sdf.drop_duplicates('subject').set_index('subject')['y_true']
    pdf['y_true'] = pdf['subject'].map(yt)
    return pdf


def patient_metrics_from_sdf(sdf, aggregator='mean', threshold=0.5):
    """Metrik level pasien dari records per-slice."""
    pdf = patient_df(sdf, aggregator)
    yp = (pdf['p'] >= threshold).astype(int)
    m = compute_metrics(pdf['y_true'].tolist(), yp.tolist(), pdf['p'].tolist())
    m['n'] = int(len(pdf))
    return m, pdf


def tune_threshold(sdf, aggregator='mean', metric='f1', step=0.02):
    """Scan threshold optimal pd data VALIDASI (JANGAN dipakai pd test — leakage!).

    Return (best_threshold, best_metric_value, curve). Untuk LVO-triage yg mengejar recall,
    ganti metric='recall' atau baca kurva di bagian 11.
    """
    pdf = patient_df(sdf, aggregator)
    best_t, best_m, curve = 0.5, -1.0, []
    for t in np.arange(0.0, 1.0 + 1e-9, step):
        yp = (pdf['p'] >= t).astype(int)
        m = compute_metrics(pdf['y_true'].tolist(), yp.tolist(), pdf['p'].tolist())
        curve.append((float(t), float(m[metric])))
        if m[metric] > best_m:
            best_t, best_m = float(t), float(m[metric])
    return best_t, best_m, curve


def save_atomic(payload, path, save_fn=None):
    """[KAGGLE] Simpan atomic: tulis ke .tmp lalu os.replace (anti-korup saat kernel di-kill).

    Pola produksi timm.CheckpointSaver._replace — wajib untuk background run Kaggle
    (time limit bisa mematikan proses di tengah torch.save).
    """
    import os as _os, tempfile as _tf
    _d = _os.path.dirname(path) or '.'
    _os.makedirs(_d, exist_ok=True)
    fd, tmp = _tf.mkstemp(dir=_d, prefix='tmp_ckpt_')
    _os.close(fd)
    try:
        (save_fn or __import__('torch').save)(payload, tmp)
        _os.replace(tmp, path)   # atomic di POSIX
    finally:
        if _os.path.exists(tmp):
            try:
                _os.remove(tmp)
            except OSError:
                pass


def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, patience=PATIENCE,
                label='', class_weights=None, seed=42, amp=AMP, monitor=MONITOR, tta=USE_TTA,
                resume=RESUME):
    """Training: warmup+cosine, AMP, grad clip, early stopping + [KAGGLE] background-safe.

    Tambahan [KAGGLE] dibanding V3 asli:
    - Tiap epoch simpan FULL-STATE `{label}_last_full.pt` (model+optimizer+scheduler+scaler+
      epoch+best_score+history) secara atomic → tahan terhadap timeout/interrupt Kaggle.
    - `resume=True` → lanjut otomatis dari `_last_full.pt` (tidak mengulang dari epoch 0).
    - `try/except KeyboardInterrupt` → simpan `_interrupted_full.pt` sebelum keluar.
    - Checkpoint BEST `{label}_best.pt` tetap format V3 asli ({'model','meta'}) agar kompatibel
      dengan `load_best_model` di bagian pipeline.
    """
    set_seed(seed)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    if WARMUP_EPOCHS > 0 and WARMUP_EPOCHS < epochs:
        warmup = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
        cosine = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - WARMUP_EPOCHS)
        scheduler = optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine],
                                                    milestones=[WARMUP_EPOCHS])
    else:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    invert = ('stage2_laterality' in label)          # flip TTA membalik kelas utk laterality
    use_amp = amp and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    history = {'train_loss': [], 'val_loss': [], 'val_img_f1': [], 'val_pat_f1': []}
    best_score, best_epoch, bad_epochs = -1.0, 0, 0
    save_path = os.path.join(OUTPUT_DIR, f'{label}_best.pt')
    last_path = os.path.join(OUTPUT_DIR, f'{label}_last_full.pt')
    start_epoch = 1

    # ---- [KAGGLE] resume dari full-state bila ada ----
    if resume and os.path.exists(last_path):
        try:
            ck = torch.load(last_path, map_location=device, weights_only=False)
            model.load_state_dict(ck['model'])
            optimizer.load_state_dict(ck['optimizer'])
            if 'scheduler' in ck and ck['scheduler'] is not None:
                try:
                    scheduler.load_state_dict(ck['scheduler'])
                except Exception as e:
                    print(f'[{label}] peringatan: scheduler state tidak cocok, lanjut tanpa itu ({e})')
            if 'scaler' in ck and ck['scaler'] is not None and use_amp:
                try:
                    scaler.load_state_dict(ck['scaler'])
                except Exception as e:
                    print(f'[{label}] peringatan: scaler state tidak cocok ({e})')
            best_score = ck.get('best_score', best_score)
            best_epoch = ck.get('best_epoch', 0)
            bad_epochs = ck.get('bad_epochs', 0)
            history = ck.get('history', history)
            start_epoch = int(ck.get('epoch', 0)) + 1
            print(f'[{label}] ▶ resume dari epoch {start_epoch} (best {monitor}={best_score:.4f} @epoch {best_epoch})')
        except Exception as e:
            print(f'[{label}] peringatan: gagal resume dari {last_path} ({e}) → mulai dari awal.')

    # ---- [KAGGLE] warm-start dari weights dataset lama (opsional, hanya bila belum ada resume) ----
    if start_epoch == 1 and LOAD_WEIGHTS_DIR:
        _w = os.path.join(LOAD_WEIGHTS_DIR, f'{label}_best.pt')
        if os.path.exists(_w):
            try:
                _ck = torch.load(_w, map_location=device, weights_only=False)
                _sd = _ck['model'] if isinstance(_ck, dict) and 'model' in _ck else _ck
                model.load_state_dict(_sd)
                print(f'[{label}] warm-start dari {_w}')
            except Exception as e:
                print(f'[{label}] peringatan: gagal warm-start ({e})')

    try:
        for epoch in range(start_epoch, epochs + 1):
            model.train()
            tot_loss, tot = 0.0, 0
            for xb, yb, _, _ in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda', enabled=use_amp):
                    logits = model(xb)
                    loss = criterion(logits, yb)
                scaler.scale(loss).backward()
                if GRAD_CLIP:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                tot_loss += loss.item() * len(yb)
                tot += len(yb)
            scheduler.step()

            val_metrics, val_recs = evaluate_model(model, val_loader, criterion=criterion,
                                                   tta=tta, invert_flip=invert)
            pa_val, _ = patient_metrics_from_sdf(scores_df(val_recs), 'mean', 0.5)
            score = pa_val['f1'] if monitor == 'patient_f1' else val_metrics['f1']
            tl = tot_loss / max(tot, 1)
            history['train_loss'].append(tl)
            history['val_loss'].append(val_metrics.get('loss', float('nan')))
            history['val_img_f1'].append(val_metrics['f1'])
            history['val_pat_f1'].append(pa_val['f1'])

            print(f'[{label}] epoch {epoch:02d}/{epochs} | train_loss={tl:.4f} '
                  f'| val_img_f1={val_metrics["f1"]:.4f} | val_pat_f1={pa_val["f1"]:.4f}', flush=True)

            if score > best_score:
                best_score, best_epoch, bad_epochs = score, epoch, 0
                save_atomic({'model': model.state_dict(),
                             'meta': dict(label=label, seed=seed, monitor=monitor,
                                          epoch=epoch, val_pat_f1=pa_val['f1'],
                                          val_img_f1=val_metrics['f1'], config=CONFIG)},
                            save_path)
            else:
                bad_epochs += 1
                # [KAGGLE] simpan full-state tiap SAVE_EVERY epoch (atomic) — walau bukan best
                if (epoch % SAVE_EVERY == 0) or (bad_epochs >= patience):
                    save_atomic({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                                 'scheduler': scheduler.state_dict(),
                                 'scaler': scaler.state_dict() if use_amp else None,
                                 'epoch': epoch, 'best_score': best_score, 'best_epoch': best_epoch,
                                 'bad_epochs': bad_epochs, 'history': history,
                                 'meta': dict(label=label, seed=seed, monitor=monitor, config=CONFIG)},
                                last_path)
                if bad_epochs >= patience:
                    print(f'  >> Early stop di epoch {epoch} (best {monitor}={best_score:.4f} di epoch {best_epoch})')
                    break
            # [KAGGLE] pastikan last_full selalu ada tiap epoch (untuk resume kapan pun)
            if epoch % SAVE_EVERY == 0:
                save_atomic({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                             'scheduler': scheduler.state_dict(),
                             'scaler': scaler.state_dict() if use_amp else None,
                             'epoch': epoch, 'best_score': best_score, 'best_epoch': best_epoch,
                             'bad_epochs': bad_epochs, 'history': history,
                             'meta': dict(label=label, seed=seed, monitor=monitor, config=CONFIG)},
                            last_path)
    except KeyboardInterrupt:
        # [KAGGLE] interrupt (stop manual / pre-timeout) → selamatkan state sebelum keluar
        try:
            save_atomic({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                         'scheduler': scheduler.state_dict(),
                         'scaler': scaler.state_dict() if use_amp else None,
                         'epoch': epoch, 'best_score': best_score, 'best_epoch': best_epoch,
                         'bad_epochs': bad_epochs, 'history': history,
                         'meta': dict(label=label, seed=seed, monitor=monitor, config=CONFIG)},
                        os.path.join(OUTPUT_DIR, f'{label}_interrupted_full.pt'))
            print(f'[{label}] ⚠ interrupted di epoch {epoch} — state tersimpan, jalankan ulang utk resume.')
        except Exception as e:
            print(f'[{label}] gagal menyimpan interrupted state: {e}')
        raise

    if os.path.exists(save_path):
        _best = torch.load(save_path, map_location=device, weights_only=False)
        model.load_state_dict(_best['model'] if isinstance(_best, dict) and 'model' in _best else _best)
    print(f'[{label}] Selesai. Best {monitor} = {best_score:.4f} (epoch {best_epoch})')
    return model, history


In [ ]:

# %% [markdown]
# ## 1. Backbone — build_model_v4 (ResNet18 / DenseNet121)

# %%
import torch
import torch.nn as nn
import timm

# Bobot RadImageNet terverifikasi (HF: Lab-Rasool/RadImageNet):
# - DenseNet121.pt = OrderedDict backbone-only TANPA classifier, prefix 'backbone.0.'.
#   Dipetakan ke 'features.' (timm). Diverifikasi: 725/725 key cocok 100%.
# - ResNet50.pt memakai penamaan Sequential flat -> TIDAK dipetakan (rapuh).
# - Official RadImageNet TIDAK menyediakan resnet18 -> resnet18 selalu timm ImageNet.
RADIMAGENET_REPO = 'Lab-Rasool/RadImageNet'
RADIMAGENET_FILES = {
    'densenet121': 'DenseNet121.pt',
}


def _load_radimagenet_weights(model, backbone_name):
    """Muat bobot RadImageNet terverifikasi. Return (matched, total).

    Raise RuntimeError bila tak ada file terverifikasi / kecocokan <90%.
    """
    from torch.hub import load_state_dict_from_url
    if backbone_name not in RADIMAGENET_FILES:
        raise RuntimeError(f'tidak ada bobot RadImageNet terverifikasi utk {backbone_name}')
    url = f'https://huggingface.co/{RADIMAGENET_REPO}/resolve/main/{RADIMAGENET_FILES[backbone_name]}'
    sd = load_state_dict_from_url(url, map_location='cpu', progress=True)
    if not isinstance(sd, dict):
        raise RuntimeError(f'format tak dikenal dari {url}: {type(sd)}')
    own = model.state_dict()
    mapped = {}
    for k, v in sd.items():
        if k.startswith('backbone.0.'):
            nk = 'features.' + k[len('backbone.0.'):]
            if nk in own and tuple(own[nk].shape) == tuple(v.shape):
                mapped[nk] = v
    res = model.load_state_dict(mapped, strict=False)
    total, matched = len(own), len(mapped)
    print(f'[BACKBONE] RadImageNet {backbone_name}: {matched}/{total} key cocok '
          f'({matched / total:.1%}), missing={len(res.missing_keys)}')
    if matched / total < 0.9:
        raise RuntimeError(f'kecocokan RadImageNet rendah ({matched}/{total})')
    return matched, total

def build_model_v4(backbone_name="resnet18", num_classes=1, dropout=0.4):
    """Create backbone with RadImageNet weights; fall back to timm ImageNet."""
    model = timm.create_model(backbone_name, pretrained=False, num_classes=0)
    try:
        _load_radimagenet_weights(model, backbone_name)
        print(f"[BACKBONE] RadImageNet loaded -> {backbone_name}")
    except Exception as e:
        print(f"[BACKBONE] RadImageNet tidak dipakai ({e})")
        print(f"[BACKBONE] fallback timm ImageNet -> {backbone_name}")
        model = timm.create_model(backbone_name, pretrained=True, num_classes=0)

    # Ganti classifier ASLI timm (resnet->fc, densenet->classifier) lewat reset_classifier.
    # (JANGAN model.head: forward timm tidak memakainya -> loss tanpa grad saat freeze.)
    model.reset_classifier(num_classes)
    feat_dim = getattr(model, "num_features", 512)
    _head = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, num_classes))
    if hasattr(model, 'fc'):
        model.fc = _head
    elif hasattr(model, 'classifier'):
        model.classifier = _head
    else:
        model.head = _head  # fallback: pastikan _is_head() di bawah mengenalinya

    def _is_head(n):
        return any(k in n for k in ('fc.', 'classifier.', 'head.'))
    backbone_params = [p for n, p in model.named_parameters() if not _is_head(n)]
    head_params = [p for n, p in model.named_parameters() if _is_head(n)]
    model.param_groups = [
        {"params": backbone_params, "lr_scale": 0.1},
        {"params": head_params, "lr_scale": 1.0},
    ]
    print(f"[BACKBONE] {backbone_name} | feat={feat_dim} | "
          f"params={sum(p.numel() for p in model.parameters()):,}")
    return model


_HEAD_KEYS = ('fc.', 'classifier.', 'head.')


def freeze_backbone(model):
    """Freeze all layers except classification head (fc/classifier/head)."""
    for n, p in model.named_parameters():
        if not any(k in n for k in _HEAD_KEYS):
            p.requires_grad = False
    print("[BACKBONE] Frozen backbone, head trainable")


def unfreeze_backbone(model):
    """Unfreeze every parameter for fine-tuning."""
    for p in model.parameters():
        p.requires_grad = True
    print("[BACKBONE] All layers unfrozen")

# %%
# Quick smoke test
if __name__ == "__main__":
    for name in ("resnet18", "densenet121"):
        m = build_model_v4(name, num_classes=1)
        freeze_backbone(m)
        unfreeze_backbone(m)





In [ ]:

# %% [markdown]
# ## 2. Siamese Laterality Network (Stage 2 only)

# %%
import torch
import torch.nn as nn

USE_SIAMESE_STAGE2 = True  # Set False to disable siamese path


class SiameseLateralityNet(nn.Module):
    """Shared encoder on hemispheres + difference features -> classifier."""

    def __init__(self, encoder, num_classes=2, dropout=0.4):
        super().__init__()
        self.encoder = encoder
        feat_dim = getattr(encoder, "num_features", 512)

        self.diff_proj = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim * 3, num_classes),  # left + right + diff
        )

    def forward(self, x):
        """Split slice vertically into hemispheres, encode, diff, classify."""
        w = x.size(-1)
        mid = w // 2
        left_hemi = x[:, :, :, mid:]     # right side of image -> left hemisphere
        right_hemi = x[:, :, :, :mid]    # left side of image -> right hemisphere

        feat_l = self.encoder(left_hemi)
        feat_r = self.encoder(right_hemi)
        feat_diff = self.diff_proj(torch.abs(feat_l - feat_r))

        combined = torch.cat([feat_l, feat_r, feat_diff], dim=-1)
        return self.classifier(combined)


def maybe_build_siamese(backbone, num_classes=2, dropout=0.4):
    """Wrap encoder in SiameseLateralityNet if USE_SIAMESE_STAGE2 is True."""
    if USE_SIAMESE_STAGE2:
        model = SiameseLateralityNet(backbone, num_classes=num_classes, dropout=dropout)
        print("[SIAMESE] Stage 2 siamese laterality network enabled")
        return model
    print("[SIAMESE] Siamese disabled - using plain backbone")
    return backbone

# %%
# Quick smoke test (encoder kecil -> vektor fitur, meniru timm num_classes=0)
if __name__ == "__main__":
    enc = torch.nn.Sequential(
        torch.nn.AdaptiveAvgPool2d((1, 1)),
        torch.nn.Flatten(),
        torch.nn.Linear(3, 512),
    )
    model = SiameseLateralityNet(enc, num_classes=2)
    dummy = torch.randn(2, 3, 128, 128)
    out = model(dummy)
    print(f"[SIAMESE] Input {tuple(dummy.shape)} -> Output {tuple(out.shape)}")




In [ ]:

# %% [markdown]
# ## 3. Test-Time Augmentation - Intensity Only (max 5 views)

# %%
import numpy as np
import torch


def tta_intensity_views(image_np, max_views=5):
    """
    Generate intensity-only augmented views from a single (H, W) or (C, H, W) array.
    No geometric transforms (no flips) to preserve laterality symmetry.
    Views: original | brightness | contrast | noise | bright+noise combo.
    """
    arr = image_np.astype(np.float32)
    views = [image_np.copy()]

    # 1) Brightness shift
    shift = np.random.uniform(0.1, 0.3) * max(arr.mean(), 1e-6)
    bright = np.clip(arr + shift, 0, arr.max() if arr.max() > 0 else 1)
    views.append(bright.astype(image_np.dtype))

    # 2) Contrast stretch
    lo, hi = float(arr.min()), float(arr.max())
    denom = hi - lo + 1e-8
    contrasted = np.clip((arr - lo) / denom * 255, 0, 255)
    views.append(contrasted.astype(np.uint8))

    # 3) Gaussian noise
    sigma = np.random.uniform(0.01, 0.03)
    noisy = np.clip(arr + np.random.normal(0, sigma, arr.shape), 0, arr.max() or 1)
    views.append(noisy.astype(image_np.dtype))

    # 4) Brightness + noise combo
    combo = np.clip(bright + np.random.normal(0, sigma * 0.6, arr.shape), 0, arr.max() or 1)
    views.append(combo.astype(image_np.dtype))

    return views[:max_views]


def invert_flip_mask(n_views):
    """Return all-False: no geometric flips in intensity-only TTA."""
    return [False] * n_views


@torch.no_grad()
def evaluate_model_v4(model, dataloader, device="cuda", max_tta=5):
    """Evaluate model with intensity-only TTA. Returns (preds, labels)."""
    model.eval()
    all_preds, all_labels = [], []

    for images, labels in dataloader:
        batch_np = images.cpu().numpy()
        for i in range(batch_np.shape[0]):
            views = tta_intensity_views(batch_np[i], max_views=max_tta)
            view_preds = []
            for v in views:
                t = torch.from_numpy(v).unsqueeze(0).float().to(device)
                view_preds.append(torch.sigmoid(model(t)).item())
            all_preds.append(float(np.mean(view_preds)))
        all_labels.extend(labels.cpu().numpy().tolist())

    return np.array(all_preds), np.array(all_labels)

# %%
# Quick smoke test
if __name__ == "__main__":
    fake_img = np.random.rand(64, 64).astype(np.float32)
    views = tta_intensity_views(fake_img, max_views=5)
    mask = invert_flip_mask(len(views))
    print(f"[TTA] Generated {len(views)} views, geometric_flip_mask={mask}")



In [ ]:

# %% [markdown]
# ## 4. Ensemble - Soft Vote Weighted by Validation AUC

# %%
import os
import gc
import numpy as np
import torch


def soft_vote_ensemble(checkpoints, dataloader, model_builder, device="cuda",
                       weight_by_auc=True):
    """
    Memory-efficient weighted soft-vote ensemble.

    Args:
        checkpoints:   list of (ckpt_path, val_auc) tuples
        dataloader:    test dataloader (batch_size can be small)
        model_builder: callable(path) -> nn.Module (loads weights internally)
        weight_by_auc: if True, weight predictions by validation AUC

    Returns:
        final_preds: np.ndarray (N_samples,)
        labels:      np.ndarray (N_samples,) or None
    """
    weights = np.array([auc for _, auc in checkpoints]) if weight_by_auc \
        else np.ones(len(checkpoints))
    norm_w = weights / weights.sum()

    all_preds = []
    labels = None

    for ckpt_path, val_auc in checkpoints:
        print(f"[ENSEMBLE] Loading {os.path.basename(ckpt_path)}  (AUC={val_auc:.4f})")
        model = model_builder(ckpt_path)
        model.to(device).eval()

        preds_buf = []
        with torch.no_grad():
            for images, lbls in dataloader:
                images = images.to(device, non_blocking=True)
                out = torch.sigmoid(model(images)).cpu().numpy().ravel()
                preds_buf.append(out)
                if labels is None:
                    labels = lbls.numpy()

        all_preds.append(np.concatenate(preds_buf))

        del model
        gc.collect()
        torch.cuda.empty_cache()

    preds_arr = np.stack(all_preds, axis=0)   # (N_models, N_samples)
    final = (preds_arr * norm_w[:, np.newaxis]).sum(axis=0)
    print(f"[ENSEMBLE] {len(checkpoints)} models -> final shape {final.shape}")
    return final, labels

# %%
# Example usage
if __name__ == "__main__":
    checkpoints = [
        ("resnet18_fold0.pt", 0.9123),
        ("densenet121_fold0.pt", 0.9201),
        ("resnet18_fold1.pt", 0.9087),
    ]
    print(f"[ENSEMBLE] Would ensemble {len(checkpoints)} checkpoints")
    w = np.array([a for _, a in checkpoints])
    print(f"[ENSEMBLE] Weights (AUC-normalised): {(w / w.sum()).round(4)}")



In [ ]:

def _get_backbone_layers(model):
    """Identifikasi layer backbone vs head pada model timm.

    Return (backbone_params, head_params) untuk optimasi terpisah.
    """
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if any(k in name for k in ['fc.', 'classifier.', 'head.']):
            head_params.append(param)
        else:
            backbone_params.append(param)
    return backbone_params, head_params


def _ema_smooth(current_ema, new_value, alpha=0.5):
    """EMA smoothing: ema = alpha * new + (1 - alpha) * ema."""
    return alpha * new_value + (1 - alpha) * current_ema


def train_model_v4(model, loaders, label, seed=42):
    """Training V4: freeze backbone -> unfreeze + layer-wise LR, AMP, grad clip.

    Fase 1 (FREEZE_EPOCHS): Freeze semua layer kecuali head/classifier.
      - LR = LR_HEAD, optimizer AdamW
    Fase 2: Unfreeze semua layer.
      - Backbone LR = LR_BACKBONE, Head LR = LR_HEAD
      - Layer-wise learning rate via parameter groups
    Fitur:
      - AdamW wd=1e-4, cosine+warmup, AMP, grad clip 1.0
      - Class-balanced focal loss (LABEL_SMOOTH, FOCAL_GAMMA)
      - Monitor = EMA-smoothed (alpha 0.5) val patient AUC
      - Early stopping patience 7
      - Best checkpoint format V3 {'model','meta'} + last_full atomic via save_atomic
      - Resume dari last_full, KeyboardInterrupt -> interrupted_full

    Args:
        model: model timm
        loaders: dict {'train': train_loader, 'val': val_loader, 'test': test_loader}
        label: str identifier untuk checkpoint naming
        seed: int random seed

    Returns:
        (model, history dict dengan train_loss, val_auc_smooth, val_pat_f1)
    """
    set_seed(seed)
    train_loader = loaders['train']
    val_loader = loaders['val']

    # Class-balanced focal loss
    criterion = ClassBalancedFocalLoss(
        num_classes=2, beta=0.99, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTH
    ).to(device)

    # Identifikasi backbone vs head
    backbone_params, head_params = _get_backbone_layers(model)

    # Optimizer: layer-wise LR
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': LR_BACKBONE},
        {'params': head_params, 'lr': LR_HEAD},
    ], weight_decay=1e-4)

    # Scheduler: warmup + cosine
    total_epochs = EPOCHS
    if WARMUP_EPOCHS > 0 and WARMUP_EPOCHS < total_epochs:
        warmup = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
        cosine = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs - WARMUP_EPOCHS)
        scheduler = optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine],
                                                    milestones=[WARMUP_EPOCHS])
    else:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)

    # AMP
    use_amp = AMP and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    # Resume
    save_path = os.path.join(OUTPUT_DIR, f'{label}_best.pt')
    last_path = os.path.join(OUTPUT_DIR, f'{label}_last_full.pt')
    start_epoch = 1
    best_score, best_epoch, bad_epochs = -1.0, 0, 0
    history = {'train_loss': [], 'val_auc_smooth': [], 'val_pat_f1': []}
    val_auc_ema = None

    if RESUME and os.path.exists(last_path):
        try:
            ck = torch.load(last_path, map_location=device, weights_only=False)
            model.load_state_dict(ck['model'])
            optimizer.load_state_dict(ck['optimizer'])
            if 'scheduler' in ck and ck['scheduler'] is not None:
                try:
                    scheduler.load_state_dict(ck['scheduler'])
                except Exception as e:
                    print(f'[{label}] peringatan: scheduler state tidak cocok ({e})')
            if 'scaler' in ck and ck['scaler'] is not None and use_amp:
                try:
                    scaler.load_state_dict(ck['scaler'])
                except Exception as e:
                    print(f'[{label}] peringatan: scaler state tidak cocok ({e})')
            best_score = ck.get('best_score', best_score)
            best_epoch = ck.get('best_epoch', 0)
            bad_epochs = ck.get('bad_epochs', 0)
            history = ck.get('history', history)
            start_epoch = int(ck.get('epoch', 0)) + 1
            val_auc_ema = ck.get('val_auc_ema', None)
            print(f'[{label}] ▶ resume dari epoch {start_epoch} (best {MONITOR}={best_score:.4f} @epoch {best_epoch})')
        except Exception as e:
            print(f'[{label}] peringatan: gagal resume dari {last_path} ({e}) → mulai dari awal.')

    # Warm-start dari weights dataset lama (opsional)
    if start_epoch == 1 and LOAD_WEIGHTS_DIR:
        _w = os.path.join(LOAD_WEIGHTS_DIR, f'{label}_best.pt')
        if os.path.exists(_w):
            try:
                _ck = torch.load(_w, map_location=device, weights_only=False)
                _sd = _ck['model'] if isinstance(_ck, dict) and 'model' in _ck else _ck
                model.load_state_dict(_sd)
                print(f'[{label}] warm-start dari {_w}')
            except Exception as e:
                print(f'[{label}] peringatan: gagal warm-start ({e})')

    # Fase 1: Freeze backbone
    if FREEZE_EPOCHS > 0:
        for name, param in model.named_parameters():
            if not any(k in name for k in ['fc.', 'classifier.', 'head.']):
                param.requires_grad = False
        print(f'[{label}] Fase 1: Freeze backbone selama {FREEZE_EPOCHS} epoch (LR_HEAD={LR_HEAD})')

    try:
        for epoch in range(start_epoch, total_epochs + 1):
            model.train()
            tot_loss, tot = 0.0, 0
            for xb, yb, _, _ in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda', enabled=use_amp):
                    logits = model(xb)
                    loss = criterion(logits, yb)
                scaler.scale(loss).backward()
                if GRAD_CLIP:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                tot_loss += loss.item() * len(yb)
                tot += len(yb)
            scheduler.step()

            # Evaluasi
            model.eval()
            val_auc_list, val_pat_f1_list = [], []
            with torch.no_grad():
                for xb, yb, subj, _ in val_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    logits = model(xb)
                    probs = torch.softmax(logits, dim=1)
                    p_pos = probs[:, 1]
                    if len(set(yb.tolist())) > 1:
                        val_auc_list.append(roc_auc_score(yb.cpu().numpy(), p_pos.cpu().numpy()))
                    yp = (p_pos >= 0.5).int()
                    if len(set(yb.tolist())) > 1:
                        val_pat_f1_list.append(f1_score(yb.cpu().numpy(), yp.cpu().numpy(), zero_division=0))

            val_auc = float(np.mean(val_auc_list)) if val_auc_list else 0.0
            val_pat_f1 = float(np.mean(val_pat_f1_list)) if val_pat_f1_list else 0.0

            # EMA-smoothed val AUC
            if val_auc_ema is None:
                val_auc_ema = val_auc
            else:
                val_auc_ema = _ema_smooth(val_auc_ema, val_auc, alpha=0.5)

            tl = tot_loss / max(tot, 1)
            history['train_loss'].append(tl)
            history['val_auc_smooth'].append(val_auc_ema)
            history['val_pat_f1'].append(val_pat_f1)

            score = val_auc_ema

            print(f'[{label}] epoch {epoch:02d}/{total_epochs} | train_loss={tl:.4f} '
                  f'| val_auc_smooth={val_auc_ema:.4f} | val_pat_f1={val_pat_f1:.4f}', flush=True)

            if score > best_score:
                best_score, best_epoch, bad_epochs = score, epoch, 0
                save_atomic({'model': model.state_dict(),
                             'meta': dict(label=label, seed=seed, monitor=MONITOR,
                                          epoch=epoch, val_auc_smooth=val_auc_ema,
                                          val_pat_f1=val_pat_f1, config=CONFIG)},
                            save_path)
            else:
                bad_epochs += 1
                if (epoch % SAVE_EVERY == 0) or (bad_epochs >= PATIENCE):
                    save_atomic({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                                 'scheduler': scheduler.state_dict(),
                                 'scaler': scaler.state_dict() if use_amp else None,
                                 'epoch': epoch, 'best_score': best_score, 'best_epoch': best_epoch,
                                 'bad_epochs': bad_epochs, 'history': history,
                                 'val_auc_ema': val_auc_ema,
                                 'meta': dict(label=label, seed=seed, monitor=MONITOR, config=CONFIG)},
                                last_path)
                if bad_epochs >= PATIENCE:
                    print(f'  >> Early stop di epoch {epoch} (best {MONITOR}={best_score:.4f} di epoch {best_epoch})')
                    break
            if epoch % SAVE_EVERY == 0:
                save_atomic({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                             'scheduler': scheduler.state_dict(),
                             'scaler': scaler.state_dict() if use_amp else None,
                             'epoch': epoch, 'best_score': best_score, 'best_epoch': best_epoch,
                             'bad_epochs': bad_epochs, 'history': history,
                             'val_auc_ema': val_auc_ema,
                             'meta': dict(label=label, seed=seed, monitor=MONITOR, config=CONFIG)},
                            last_path)

        # SWA (Stochastic Weight Averaging)
        if SWA and hasattr(torch.optim.swa_utils, 'AveragedModel'):
            try:
                swa_model = torch.optim.swa_utils.AveragedModel(model)
                swa_scheduler = torch.optim.swa_utils.SWALR(optimizer, swa_lr=LR_BACKBONE)
                for _ in range(10):
                    swa_model.update_parameters(model)
                    swa_scheduler.step()
                model = swa_model
                print(f'[{label}] SWA diterapkan.')
            except Exception as e:
                print(f'[{label}] SWA gagal: {e}')

    except KeyboardInterrupt:
        try:
            save_atomic({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                         'scheduler': scheduler.state_dict(),
                         'scaler': scaler.state_dict() if use_amp else None,
                         'epoch': epoch, 'best_score': best_score, 'best_epoch': best_epoch,
                         'bad_epochs': bad_epochs, 'history': history,
                         'val_auc_ema': val_auc_ema,
                         'meta': dict(label=label, seed=seed, monitor=MONITOR, config=CONFIG)},
                        os.path.join(OUTPUT_DIR, f'{label}_interrupted_full.pt'))
            print(f'[{label}] ⚠ interrupted di epoch {epoch} — state tersimpan, jalankan ulang utk resume.')
        except Exception as e:
            print(f'[{label}] gagal menyimpan interrupted state: {e}')
        raise

    # Load best model
    if os.path.exists(save_path):
        _best = torch.load(save_path, map_location=device, weights_only=False)
        model.load_state_dict(_best['model'] if isinstance(_best, dict) and 'model' in _best else _best)
    print(f'[{label}] Selesai. Best {MONITOR} = {best_score:.4f} (epoch {best_epoch})')
    return model, history



In [ ]:
# ============================================================
#  WIRING V4 — rakit stage → model → loaders → train → eval
#  (menggantikan run_ablation V3; memakai train_model_v4 + build_model_v4)
# ============================================================
def build_stage_model(stage, bb):
    """Model per stage: siamese utk stage2, backbone biasa utk lainnya (2 kelas)."""
    if stage == 'stage2_laterality' and USE_SIAMESE_STAGE2:
        enc = timm.create_model(bb, pretrained=True, num_classes=0)
        m = SiameseLateralityNet(enc, num_classes=2, dropout=DROPOUT_HEAD)
        print(f'[V4] stage2 siamese encoder={bb}')
    else:
        m = build_model_v4(bb, num_classes=2, dropout=DROPOUT_HEAD)
    return m.to(device)


def run_ablation_v4(backbones=None, seeds=None, stages=None, epochs=None, tta=USE_TTA):
    """Ablation V4 end-to-end (seleksi memakai val_scores; TEST hanya dilaporkan sekali)."""
    backbones = backbones or BACKBONES
    seeds = seeds or SEEDS
    stages = stages or STAGES
    epochs = epochs or EPOCHS
    results, t0 = {}, time.time()
    for stage in stages:
        print(f'\n{"="*72}\nSTAGE {stage}\n{"="*72}')
        tr_ld, va_ld, te_ld = make_loaders_v4(stage)
        print(f'Citra train: {len(tr_ld.dataset)} | val: {len(va_ld.dataset)} | test: {len(te_ld.dataset)}')
        invert = (stage == 'stage2_laterality')
        results[stage] = {}
        for bb in backbones:
            results[stage][bb] = {}
            for seed in seeds:
                model = build_stage_model(stage, bb)
                model, hist = train_model_v4(model, {'train': tr_ld, 'val': va_ld, 'test': te_ld},
                                              label=f'{stage}_{bb}_s{seed}', seed=seed)
                va_m, va_r = evaluate_model(model, va_ld, tta=tta, invert_flip=invert)
                te_m, te_r = evaluate_model(model, te_ld, tta=tta, invert_flip=invert)
                results[stage][bb][seed] = {
                    'val_img': va_m, 'val_scores': scores_df(va_r),
                    'test': te_m, 'test_scores': scores_df(te_r), 'history': hist}
                print(f'  ✔ {bb} (seed={seed}) | VAL img_f1={va_m["f1"]:.4f} | TEST img_f1={te_m["f1"]:.4f}')
    print(f'\n⏱ Total waktu: {(time.time() - t0) / 60:.1f} menit')
    return results



### 8. Ablation Runner & Penyimpanan Hasil

Framework ablation (stage × backbone × seed) pada fixed split yang sama:

- Setiap run menyimpan: metrik **val** (`val_img`, `val_patient`) + **`val_scores`** dan **`test_scores`**
  (DataFrame per-slice) — dipakai utk seleksi & tuning keputusan **di val** (tanpa leakage).
- Metrik default level pasien (mean, thr 0.5) tetap dilaporkan agar bisa dibandingkan dgn V2.
- Otomatis menyimpan JSON/CSV/Markdown + **snapshot konfigurasi**.

In [ ]:
# [KAGGLE] Helper background-run: arsip weights + status sesi
import time as _time
_SESS_T0 = _time.time()

def kaggle_session_status():
    el = (_time.time() - _SESS_T0) / 3600
    print(f'Sesi berjalan {el:.2f} jam (limit GPU Kaggle ≈ 12 jam/sesi, kuota ≈ 30 jam/minggu).')
    print(f'OUTPUT_DIR: {OUTPUT_DIR}')
    pts = sorted(__import__('glob').glob(os.path.join(OUTPUT_DIR, '*.pt')))
    print(f'Checkpoint (*.pt): {len(pts)} file')
    for p in pts:
        print(f'  - {os.path.basename(p)} ({os.path.getsize(p)/1e6:.1f} MB)')

def zip_kaggle_outputs(zip_name='/kaggle/working/ncct_v4_weights.zip'):
    """[KAGGLE] Arsipkan SEMUA weights + keputusan + laporan ke 1 zip di /kaggle/working.

    Jalankan di cell terakhir (atau kapan pun). File muncul di tab Output Kaggle →
    download langsung, atau Save Version → Commit to Dataset untuk reuse.
    """
    import zipfile, glob
    pats = ['*.pt', '*.json', '*.csv', '*.md']
    files = []
    for p in pats:
        files += glob.glob(os.path.join(OUTPUT_DIR, p))
    if not files:
        print(f'Tidak ada file di {OUTPUT_DIR} — jalankan training dulu.')
        return None
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fp in sorted(files):
            zf.write(fp, arcname=f'outputs_v4/{os.path.basename(fp)}')
    print(f'✔ {zip_name} ({os.path.getsize(zip_name)/1e6:.1f} MB, {len(files)} file)')
    print('Download dari tab Output (kanan) → atau Save Version → Commit to Dataset.')
    return zip_name

kaggle_session_status()



### 9. Menjalankan Ablation

> **Estimasi waktu (Kaggle GPU P100/T4):** 1 epoch stage1 ≈ 1.5–4 menit tergantung backbone (GPU Kaggle).\n> **[KAGGLE]** Browser boleh ditutup — training lanjut di server. Batas sesi ≈ 12 jam; bila terputus, jalankan ulang notebook (auto-resume dari `*_last_full.pt`). Setelah selesai jalankan cell 17 + **Save Version** agar weights tersimpan.
> Total ≈ jumlah stage × backbone × seed × epoch efektif (early stopping).
> Untuk uji cepat, batasi variabel `RUN_*` di bawah (`RUN_EPOCHS=2` misalnya), lalu jalankan
> ulang dengan epoch penuh utk hasil final.

### 17. [KAGGLE] Arsip & Simpan Weights

Cell ini membuat `ncct_v4_weights.zip` di `/kaggle/working/` lalu instruksi persistensi (Save Version / Commit to Dataset).


In [ ]:
# ============================================================
#  KONFIGURASI RUN V4
# ============================================================
RUN_STAGES    = STAGES            # mis. ['stage1_lvo'] utk coba 1 stage dulu
RUN_BACKBONES = BACKBONES         # ['resnet18', 'densenet121']
RUN_SEEDS     = SEEDS             # [42, 43, 44]
RUN_EPOCHS    = EPOCHS            # dipakai di dalam train_model_v4 via EPOCHS global

results = run_ablation_v4(backbones=RUN_BACKBONES, seeds=RUN_SEEDS,
                          stages=RUN_STAGES, epochs=RUN_EPOCHS)



In [ ]:
# ===== Ringkasan V4: mean±std antar seed (VAL img + TEST img F1) + simpan =====
def _mstd(vals):
    vals = [v for v in vals if v is not None and not (isinstance(v, float) and np.isnan(v))]
    return f'{np.mean(vals):.4f}±{np.std(vals):.4f}' if vals else 'N/A'

for level in ['val_img', 'test']:
    print(f'\n### V4 — level **{level}** (mean±std antar seed)')
    print('| Stage | Backbone | F1 | AUC |')
    print('|---|---|---|---|')
    for stage in results:
        for bb in results[stage]:
            seeds = sorted(results[stage][bb])
            print(f'| {stage} | {bb} | {_mstd([results[stage][bb][s][level]["f1"] for s in seeds])} '
                  f'| {_mstd([results[stage][bb][s][level]["auc"] for s in seeds])} |')

ts = time.strftime('%Y%m%d_%H%M%S')
STRIP = {'history', 'val_scores', 'test_scores'}
slim = {st: {bb: {str(sd): {k: v for k, v in r.items() if k not in STRIP}
                   for sd, r in sdict.items()} for bb, sdict in bdict.items()}
        for st, bdict in results.items()}
with open(os.path.join(OUTPUT_DIR, f'results_v4_{ts}.json'), 'w') as f:
    json.dump(slim, f, indent=2, default=str)
with open(os.path.join(OUTPUT_DIR, 'config_snapshot.json'), 'w') as f:
    json.dump(CONFIG, f, indent=2, default=str)
print('\nTersimpan: results_v4_%s.json + config_snapshot.json' % ts)



### 11. Seleksi V4 — Joint (backbone, agregator, threshold) HANYA dari VALIDASI

Urutan cell: `THRESHOLD` → `SELECT` → `PIPELINE` → `REPORT`.
Aturan anti-overfit-val (pelajaran V3: thr 0.14/0.72 ekstrem, `max` degenerat recall=1.0):
- agregator tanpa `max`; tolak konfigurasi degenerat (recall≈1 & precision<0.5)
- threshold di-clamp per stage + butuh gain ≥0.02 vs 0.5, else 0.5
- tie-break: agregator sederhana + threshold dekat 0.5



In [ ]:

# ============================================================
#  THRESHOLD — scan 0.10–0.90 step 0.05 + moving-average smoothing (w=3)
#  clamp: s1 ∈ [0.10, 0.50] · s2/s3 ∈ [0.30, 0.70] · min gain 0.02 vs 0.5
#  Standalone: butuh numpy + sklearn saja (pdf = kolom p, y_true).
# ============================================================
def tune_threshold_v4(pdf, stage, metric='f1', step=0.05, window=3, min_gain=0.02):
    """Scan threshold pd VAL (no leakage) utk patient-df {p, y_true}.

    - scan t ∈ [0.10, 0.90] step 0.05; kurva di-smooth moving-average (window=3)
    - clamp rentang per stage: stage1 [0.10, 0.50], stage2/3 [0.30, 0.70]
    - wajib gain >= min_gain vs baseline 0.5, else fallback 0.5
    Return (best_t, best_f1, curve) — curve = [(t, f1_raw, f1_smooth)].
    """
    yt = pdf['y_true'].values
    p = pdf['p'].values
    lo, hi = (0.10, 0.50) if stage == 'stage1_lvo' else (0.30, 0.70)
    ts = np.arange(0.10, 0.90 + 1e-9, step)
    raw = np.array([f1_score(yt, (p >= t).astype(int), zero_division=0) for t in ts])
    sm = np.convolve(raw, np.ones(window) / window, mode='valid')   # ts_sm = ts[1:-1]
    ts_sm = ts[1:-1]
    best_t = float(np.clip(ts_sm[int(np.argmax(sm))], lo, hi))
    best_f1 = f1_score(yt, (p >= best_t).astype(int), zero_division=0)
    base = f1_score(yt, (p >= 0.5).astype(int), zero_division=0)
    if best_f1 - base < min_gain:
        best_t, best_f1 = 0.5, base
    curve = [(float(t), float(r), float(s)) for t, r, s in zip(ts_sm, raw[1:-1], sm)]
    return best_t, best_f1, curve



In [ ]:

# ============================================================
#  SELECT — joint (backbone, aggregator, threshold) dari VAL only
#  mean over seeds · aggregators mean/median/topk/attention/vote (NO max)
#  reject degenerate (recall==1 & precision<0.5) · tie-break simpler + closer 0.5
#  Requires tune_threshold_v4 (SECTION_THRESHOLD) & `results` (run_ablation_v4).
# ============================================================
AGG_V4 = ['mean', 'median', 'topk', 'attention', 'vote']          # no 'max'
AGG_SIMPLE = {'mean': 0, 'median': 1, 'vote': 2, 'topk': 3, 'attention': 4}


def patient_df_v4(sdf, aggregator='mean', k=3):
    """Agregasi slice → 1 baris/pasien (prob kelas-1). V4: +topk, +attention, no max."""
    grp = sdf.groupby('subject')['y_prob']
    if aggregator == 'median':
        p = grp.median()
    elif aggregator == 'topk':
        p = grp.apply(lambda x: float(np.sort(x.values)[-k:].mean()))
    elif aggregator == 'attention':
        p = grp.apply(lambda x: float(np.average(x.values, weights=np.abs(x.values - 0.5) + 1e-9)))
    elif aggregator == 'vote':
        p = grp.apply(lambda x: float((x >= 0.5).mean()))
    else:
        p = grp.mean()
    pdf = p.reset_index().rename(columns={'y_prob': 'p'})
    yt = sdf.drop_duplicates('subject').set_index('subject')['y_true']
    pdf['y_true'] = pdf['subject'].map(yt)
    return pdf


def _combo_val(results, stage, bb, agg, seeds):
    """Mean-over-seeds F1/P/R pd VAL utk (bb, agg); threshold di-tune joint pd pooled."""
    pdfs = [patient_df_v4(results[stage][bb][s]['val_scores'], agg) for s in seeds]
    thr, _, _ = tune_threshold_v4(pd.concat(pdfs), stage)
    f1s, precs, recs = [], [], []
    for pdf in pdfs:
        m = compute_metrics(pdf['y_true'].tolist(), (pdf['p'] >= thr).astype(int).tolist(), pdf['p'].tolist())
        f1s.append(m['f1']); precs.append(m['precision']); recs.append(m['recall'])
    return {'f1': float(np.mean(f1s)), 'precision': float(np.mean(precs)),
            'recall': float(np.mean(recs)), 'threshold': thr}


def select_decisions_v4(results, seeds=None):
    """Joint pick (backbone, aggregator, threshold) — HANYA VAL, tanpa leakage."""
    if seeds is None:
        st0 = list(results)[0]
        seeds = sorted(results[st0][list(results[st0])[0]])
    decisions = {}
    for stage in results:
        combos = []
        for bb in results[stage]:
            for agg in AGG_V4:
                r = _combo_val(results, stage, bb, agg, seeds)
                combos.append((r['f1'], -AGG_SIMPLE[agg], -abs(r['threshold'] - 0.5), bb, agg, r))
        valid = [c for c in combos if not (c[5]['recall'] >= 0.999 and c[5]['precision'] < 0.5)]
        cands = valid or combos
        cands.sort(key=lambda c: (c[0], c[1], c[2]), reverse=True)
        f1, _, _, bb, agg, r = cands[0]
        decisions[stage] = {'backbone': bb, 'aggregator': agg, 'threshold': r['threshold'],
                            'val_f1': f1, 'val_precision': r['precision'], 'val_recall': r['recall']}
        print(f'{stage:22s} → bb={bb:16s} agg={agg:10s} thr={r["threshold"]:.2f} F1={f1:.4f} (P={r["precision"]:.3f} R={r["recall"]:.3f})')
    return decisions


# Jalankan setelah run_ablation_v4:  DECISIONS_V4 = select_decisions_v4(results)



In [ ]:
# ===== Keputusan V4 (HANYA dari VAL) =====
DECISIONS_V4 = select_decisions_v4(results)
with open(os.path.join(OUTPUT_DIR, 'decisions_v4.json'), 'w') as f:
    json.dump(DECISIONS_V4, f, indent=2)
print('\nKeputusan tersimpan: decisions_v4.json')
print('\n⚠ Metrik TEST final dilaporkan SEKALI di cell pipeline — tidak dipakai utk seleksi.')



In [ ]:

# ============================================================
#  PIPELINE — run_ablation_v4 · pred-gate vs oracle-gate · bootstrap 95%CI
#  save decisions_v4.json + csvs · prune_checkpoints (keep best+last)
# ============================================================
def run_ablation_v4(backbones=None, seeds=None, stages=None, epochs=None, tta=USE_TTA):
    """Ablation V4: train/eval per (stage, backbone, seed) → val_scores utk seleksi."""
    backbones, seeds, stages, epochs = backbones or BACKBONES, seeds or SEEDS, stages or STAGES, epochs or EPOCHS
    results, t0 = {}, time.time()
    for stage in stages:
        cw, _ = class_weights_for(stage)
        tr_ld, va_ld, te_ld = make_loaders(stage)
        invert = (stage == 'stage2_laterality')
        results[stage] = {}
        for bb in backbones:
            results[stage][bb] = {}
            for seed in seeds:
                set_seed(seed)
                model = build_model(2, bb).to(device)
                model, hist = train_model(model, tr_ld, va_ld, epochs=epochs, label=f'{stage}_{bb}', class_weights=cw, seed=seed)
                va_m, va_r = evaluate_model(model, va_ld, tta=tta, invert_flip=invert)
                te_m, te_r = evaluate_model(model, te_ld, tta=tta, invert_flip=invert)
                results[stage][bb][seed] = {'val_img': va_m, 'val_scores': scores_df(va_r), 'test': te_m, 'test_scores': scores_df(te_r), 'history': hist}
    print(f'⏱ {(time.time() - t0) / 60:.1f} menit')
    return results


def evaluate_pipeline_v4(results, decisions, seed=None):
    """Prediksi-gate vs oracle-gate pd TEST (dipakai SEKALI). Return (pipe, rows)."""
    d1 = decisions['stage1_lvo']
    seed = seed or sorted(results['stage1_lvo'][d1['backbone']])[0]
    pipe = patient_df_v4(results['stage1_lvo'][d1['backbone']][seed]['test_scores'], d1['aggregator']).rename(columns={'p': 'p_lvo', 'y_true': 'gt_lvo'})
    pipe['pred_lvo'] = (pipe['p_lvo'] >= d1['threshold']).astype(int)
    rows = [('stage1_lvo', 'pred-gate', compute_metrics(pipe['gt_lvo'].tolist(), pipe['pred_lvo'].tolist(), pipe['p_lvo'].tolist()), len(pipe))]
    for st, pcol, gcol in [('stage2_laterality', 'p_right', 'gt_lat'), ('stage3_etiology', 'p_cardio', 'gt_etio')]:
        pdf = patient_df_v4(results[st][decisions[st]['backbone']][seed]['test_scores'], decisions[st]['aggregator'])
        pipe = pipe.merge(pdf.rename(columns={'p': pcol, 'y_true': gcol}), on='subject', how='left')
        pipe[f'pred_{st}'] = (pipe[pcol] >= decisions[st]['threshold']).astype(int)
        for gate, mask in [('pred-gate', pipe['pred_lvo'] == 1), ('oracle-gate', pipe['gt_lvo'] == 1)]:
            sub = pipe[mask].dropna(subset=[pcol, gcol])
            if len(sub) == 0:
                continue
            m = compute_metrics(sub[gcol].astype(int).tolist(), (sub[pcol] >= decisions[st]['threshold']).astype(int).tolist(), sub[pcol].astype(float).tolist())
            rows.append((st, gate, m, len(sub)))
    return pipe, rows


def bootstrap_ci_v4(y_true, p, thr=0.5, n_boot=2000, seed=0):
    """Bootstrap 95% CI F1 & AUC — resampling per SUBJEK."""
    y_true, p = np.asarray(y_true), np.asarray(p, dtype=float)
    rng, idx = np.random.default_rng(seed), np.arange(len(y_true))
    f1s, aucs = [], []
    for _ in range(n_boot):
        s = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y_true[s])) < 2:
            continue
        f1s.append(f1_score(y_true[s], (p[s] >= thr).astype(int), zero_division=0))
        aucs.append(roc_auc_score(y_true[s], p[s]))
    return ((np.percentile(f1s, 2.5), np.percentile(f1s, 97.5)), (np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)), len(y_true))


def prune_checkpoints(out_dir=OUTPUT_DIR, keep=('_best.pt', '_last_full.pt')):
    """Hapus checkpoint selain best & last (hemat disk Kaggle)."""
    removed = [os.path.basename(fp) for fp in glob.glob(os.path.join(out_dir, '*.pt')) if not fp.endswith(keep)]
    for fp in removed:
        os.remove(os.path.join(out_dir, fp))
    print(f'Pruned {len(removed)} checkpoint: {removed}')
    return removed

def run_pipeline_v4(results, decisions, out_dir=OUTPUT_DIR, n_boot=2000):
    """End-to-end: evaluasi pipeline + bootstrap CI + simpan decisions_v4.json & csvs."""
    os.makedirs(out_dir, exist_ok=True)
    pipe, rows = evaluate_pipeline_v4(results, decisions)
    with open(os.path.join(out_dir, 'decisions_v4.json'), 'w') as f:
        json.dump(decisions, f, indent=2)
    pd.DataFrame([{'stage': s, **d} for s, d in decisions.items()]).to_csv(os.path.join(out_dir, 'decisions_summary_v4.csv'), index=False)
    pd.DataFrame([{'stage': st, 'gate': g, 'n': n, **{k: round(v, 4) for k, v in m.items()}} for st, g, m, n in rows]).to_csv(os.path.join(out_dir, 'pipeline_metrics_v4.csv'), index=False)
    pipe.to_csv(os.path.join(out_dir, 'patient_pipeline_predictions_v4.csv'), index=False)
    ci_rows = []
    for st, pcol, gcol in [('stage1_lvo', 'p_lvo', 'gt_lvo'), ('stage2_laterality', 'p_right', 'gt_lat'), ('stage3_etiology', 'p_cardio', 'gt_etio')]:
        sub = pipe.dropna(subset=[pcol, gcol])
        sub = sub[sub['pred_lvo'] == 1] if st != 'stage1_lvo' else sub
        if len(sub) == 0:
            continue
        (flo, fhi), (alo, ahi), n = bootstrap_ci_v4(sub[gcol].astype(int).values, sub[pcol].astype(float).values, thr=decisions[st]['threshold'], n_boot=n_boot)
        ci_rows.append({'stage': st, 'n': n, 'f1_lo': flo, 'f1_hi': fhi, 'auc_lo': alo, 'auc_hi': ahi})
    pd.DataFrame(ci_rows).to_csv(os.path.join(out_dir, 'patient_bootstrap_ci_v4.csv'), index=False)
    print('Saved: decisions_v4.json, decisions_summary_v4.csv, pipeline_metrics_v4.csv, patient_pipeline_predictions_v4.csv, patient_bootstrap_ci_v4.csv')
    return pipe, rows, pd.DataFrame(ci_rows)
# Jalankan: results = run_ablation_v4() → DECISIONS_V4 = select_decisions_v4(results) → pipe, rows, ci = run_pipeline_v4(results, DECISIONS_V4) → prune_checkpoints()



In [ ]:
# ===== Evaluasi pipeline V4 pada TEST (1x pakai) =====
pipe, rows, ci = run_pipeline_v4(results, DECISIONS_V4)
print('\n===== PIPELINE LEVEL PASIEN — TEST (1x pakai) =====')
for st, g, m, n in rows:
    print(f'  {st:22s} ({g:11s}): acc={m["accuracy"]:.3f} prec={m["precision"]:.3f} '
          f'rec={m["recall"]:.3f} f1={m["f1"]:.3f} auc={m["auc"]:.3f} (n={n})')
print('\n===== Bootstrap 95% CI =====')
print(ci.to_string(index=False))
prune_checkpoints()



In [ ]:

# ============================================================
#  REPORT — training curves + threshold curves + V3-vs-V4 table
#  TEST dilaporkan SEKALI di sini (tidak dipakai utk seleksi apa pun)
# ============================================================
def plot_training_curves(results, stages=None, backbones=None, seed=None):
    """Kurva training (loss + val patient F1) per stage — dari history train_model."""
    stages = stages or list(results)
    bb0 = list(results[stages[0]])
    backbones = backbones or bb0
    seed = seed or sorted(results[stages[0]][bb0[0]])[0]
    fig, axes = plt.subplots(len(stages), 2, figsize=(12, 3.5 * len(stages)))
    axes = np.atleast_2d(axes)
    for i, st in enumerate(stages):
        for bb in backbones:
            h = results[st][bb][seed]['history']
            ep = np.arange(1, len(h['train_loss']) + 1)
            axes[i, 0].plot(ep, h['train_loss'], label=f'{bb} train')
            axes[i, 0].plot(ep, h['val_auc_smooth'], ls='--', label=f'{bb} val_auc')
            axes[i, 1].plot(ep, h['val_pat_f1'], label=bb)
        axes[i, 0].set_title(f'{st} — loss & val AUC'); axes[i, 0].legend(fontsize=8)
        axes[i, 1].set_title(f'{st} — val patient F1'); axes[i, 1].legend(fontsize=8)
    plt.tight_layout(); plt.show()


def plot_threshold_curves(results, decisions, seeds=None):
    """Kurva F1 vs threshold pd VAL utk (bb, agg) terpilih — smoothing terlihat."""
    st0 = list(results)[0]
    seeds = seeds or sorted(results[st0][list(results[st0])[0]])
    fig, axes = plt.subplots(1, len(results), figsize=(5.5 * len(results), 4))
    axes = np.atleast_1d(axes)
    for ax, stage in zip(axes, results):
        d = decisions[stage]
        pdfs = [patient_df_v4(results[stage][d['backbone']][s]['val_scores'], d['aggregator']) for s in seeds]
        _, _, curve = tune_threshold_v4(pd.concat(pdfs), stage)
        ts = [c[0] for c in curve]; raw = [c[1] for c in curve]; sm = [c[2] for c in curve]
        ax.plot(ts, raw, '-o', ms=3, alpha=0.5, label='raw')
        ax.plot(ts, sm, '-o', ms=3, label='smoothed')
        ax.axvline(d['threshold'], ls='--', color='red', alpha=0.7)
        ax.set_title(f'{stage} — F1 vs threshold (VAL)\nthr={d["threshold"]:.2f}')
        ax.set_xlabel('threshold'); ax.set_ylabel('F1'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


def v3_vs_v4_table(decisions_v3, decisions_v4, rows_v4):
    """Tabel V3 vs V4: keputusan (VAL) + metrik TEST (dilaporkan SEKALI)."""
    dec = pd.DataFrame([{'stage': s, 'v3': f"{d['backbone']}/{d['aggregator']}/{d['threshold']:.2f}",
                         'v4': f"{decisions_v4[s]['backbone']}/{decisions_v4[s]['aggregator']}/{decisions_v4[s]['threshold']:.2f}"}
                        for s, d in decisions_v3.items()])
    met = pd.DataFrame([{'stage': st, 'gate': g, 'n': n, 'f1': round(m['f1'], 4),
                         'precision': round(m['precision'], 4), 'recall': round(m['recall'], 4)}
                        for st, g, m, n in rows_v4])
    print('===== KEPUTUSAN V3 vs V4 (seleksi VAL) =====')
    print(dec.to_string(index=False))
    print('===== METRIK TEST — dilaporkan SEKALI (prediksi-gate) =====')
    print(met.to_string(index=False))
    return dec, met


# Jalankan: plot_training_curves(results) → plot_threshold_curves(results, DECISIONS_V4)
#           → v3_vs_v4_table(DECISIONS, DECISIONS_V4, rows)   # DECISIONS = keputusan V3 lama



In [ ]:
# ===== Kurva V4 =====
plot_training_curves(results)
plot_threshold_curves(results, DECISIONS_V4)



In [ ]:
# [KAGGLE] 17. Arsip & Simpan Weights V4 (WAJIB dijalankan terakhir)
kaggle_session_status()
zip_path = zip_kaggle_outputs('/kaggle/working/ncct_v4_weights.zip')
print()
print('NEXT (di Kaggle):')
print('  1. Save Version → Save & Run All (Commit)')
print('  2. Buka tab Output → download ncct_v4_weights.zip, ATAU: Output → ... → Commit to Dataset')
print('  3. Sesi berikutnya: Add Input dataset weights → isi LOAD_WEIGHTS_DIR=/kaggle/input/<slug> → RESUME=True')
print()
print('Ambil output via CLI lokal:  kaggle kernels output <user>/<slug> -p ./ncct_v4_out')

